# Проект. Исследование стартапов

- Автор: Айриян Луиза 
- дата: 03.04.2026

## Введение

## Цели и задачи

1. Провести предобработку данных, убрать дубликаты и пропуски, проверить корректность числовых и временных значений.
2. Выделить группы компаний по срокам финансирования и сравнить их по количеству и объёму инвестиций.
3. Классифицировать сегменты рынка на массовые, средние и нишевые и учесть это в дальнейшем анализе.
4. Определить типичные и аномальные значения объёмов финансирования, исключить выбросы и ограничить период исследования.
5. Сравнить популярность и объёмы разных типов финансирования.
6. Проанализировать динамику раундов и объёмов инвестиций по годам, а также изменения в массовых сегментах рынка.
7. Рассчитать долю возврата средств для разных типов финансирования и оценить её устойчивость.
8. Подвести итоговые выводы и дать рекомендации, куда и каким образом было бы целесообразно инвестировать, если бы на дворе был 2015 год.

## Шаг 1. Знакомство с данными: загрузка и предобработка

Датасет получен из базы данных стартапов.

Название основного датасета — `cb_investments.zip`. Внутри архива один файл — `cb_investments.csv`.

Описание данных:
* `name` — название компании.
* `homepage_url` — ссылка на сайт компании.
* `category_list` — категории, в которых работает компания. Указываются через `|`.
* `market` — основной рынок или отрасль компании.
* `funding_total_usd` — общий объём привлечённых инвестиций в долларах США.
* `status` — текущий статус компании, например `operating`, `closed` и так далее.
* `country_code` — код страны, например USA.
* `state_code` — код штата или региона, например, CA.
* `region` — регион, например, SF Bay Area.
* `city` — город, в котором расположена компания.
* `funding_rounds` — общее число раундов финансирования.
* `participants` — число участников в раундах финансирования.
* `founded_at` — дата основания компании.
* `founded_month` — месяц основания в формате `YYYY-MM`.
* `founded_quarter` — квартал основания в формате `YYYY-QN`.
* `founded_year` — год основания.
* `first_funding_at` — дата первого финансирования.
* `mid_funding_at` — дата среднего по времени раунда финансирования.
* `last_funding_at` — дата последнего финансирования.
* `seed` — сумма инвестиций на посевной стадии.
* `venture` — сумма венчурных инвестиций.
* `equity_crowdfunding` — сумма, привлечённая через долевой краудфандинг.
* `undisclosed` — сумма финансирования нераскрытого типа.
* `convertible_note` — сумма инвестиций через конвертируемые займы.
* `debt_financing` — сумма долгового финансирования.
* `angel` — сумма инвестиций от бизнес-ангелов.
* `grant` — сумма полученных грантов.
* `private_equity` — сумма инвестиций в виде прямых (частных) вложений.
* `post_ipo_equity` — сумма финансирования после IPO.
* `post_ipo_debt` — сумма долгового финансирования после IPO.
* `secondary_market` — сумма сделок на вторичном рынке.
* `product_crowdfunding` — сумма, привлечённая через продуктовый краудфандинг.
* `round_A` — `round_H` — сумма инвестиций в соответствующем раунде.

Название дополнительного датасета — `cb_returns.csv`. Он содержит суммы возвратов по типам финансирования в миллионах долларов по годам.

Описание данных:
* `year` — год возврата средств.
* `seed` — сумма возвратов от посевных инвестиций.
* `venture` — сумма возвратов от венчурных инвестиций.
* `equity_crowdfunding` — сумма, возвращённая по долевому краудфандингу.
* `undisclosed` — сумма возвратов нераскрытого типа.
* `convertible_note` — сумма возвратов через конвертируемые займы.
* `debt_financing` — сумма возвратов от долгового финансирования.
* `angel` — сумма возвратов бизнес-ангелам.
* `grant` — сумма возвратов по грантам.
* `private_equity` — сумма возвратов прямых (частных) вложений.
* `post_ipo_equity` — сумма возвратов от IPO.
* `post_ipo_debt` — сумма возвратов от долгового IPO.
* `secondary_market` — сумма возвратов от сделок на вторичном рынке.
* `product_crowdfunding` — сумма возвратов по продуктовому краудфандингу.

## Структура проекта
1. Загрузка данных и знакомство с ними 
2. Инжиниринг признаков 
3. Работа с выбросами 
4. Анализ динамики 
5. Итоговый вывод и рекомендации 

Файлы находятся в папке `datasets`, если вы выполняете работу на платформе. В случае, если вы делаете работу локально, доступ к файлам в папке можно получить по адресу `https://code.s3.yandex.net/datasets/` + имя файла.

### 1.1. Вывод общей информации

Загрузите необходимые для работы библиотеки.

Совет: если вы неоднократно используете какой-либо код, вынесите его в начало проекта в виде функций.

In [1]:
!pip install phik

In [2]:
# Импортируем библиотеки
import pandas as pd

# Загружаем библиотеки для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

# Загружаем библиотеку для расчёта коэффициента корреляции phi_k
from phik import phik_matrix

ModuleNotFoundError: No module named 'scipy.stats._mvn'

Загрузите все данные по проекту.

Совет: данные из zip-архива можно загрузить следующим кодом:

`df = pd.read_csv("https://code.s3.yandex.net/datasets/cb_investments.zip", sep=';', low_memory=False)`

In [ ]:
# Загружаем данные 
invest_df = pd.read_csv("https://code.s3.yandex.net/datasets/cb_investments.zip", sep=';', low_memory=False)
returns_df = pd.read_csv("https://code.s3.yandex.net/datasets/cb_returns.csv")

In [ ]:
# Выводим первые 5 строк 
invest_df.head()
returns_df.head()

In [ ]:
# Выводим информацию о датафрейме 
invest_df.info()
returns_df.info()

In [ ]:
# Процент пропущенных значений 
display((invest_df.isna().sum() / len(invest_df) * 100).round(2).sort_values(ascending=False))

In [ ]:
# Поиск полных дубликатов строк
display(f"Количество полных дубликатов: {invest_df.duplicated().sum()}")


In [ ]:
# Проверка уникальных значений в ключевых категориальных полях 
print("\n--- Уникальные статусы компаний ---")
print(invest_df['status'].unique())


In [ ]:
# Описание числовых данных (аномалии, отрицательные значения и выбросов)
cols_to_check = [ 'funding_rounds', 'participants', 'seed', 'venture', 'angel']
display(invest_df[cols_to_check].describe())


Выведите информацию, которая необходима для принятия решений о предобработке.

В датасете более 54 294 строк и 40 столбцов. Названия столбцов и их содержание в целом соответствуют описанию. Присутствуют как категориальные признаки (рынки, страны, города), так и количественные (суммы раундов от Seed до Round H). 
- Количество пропусков: В столбцах, связанных с датами (`founded_at`, `founded_month` и др.) и географией (`state_code`, `region`), наблюдается значительное количество пропусков (до 20-30%).
- Отраслевые признаки: В `category_list` и `market` также есть пустые значения.
- Столбцы `founded_at`, `first_funding_at`, `last_funding_at` и др. распознаны как `object`. Их необходимо привести к формату `datetime`, чтобы корректно считать возраст компаний или периоды между раундами.
- Столбцы с суммами инвестиций (`funding_total_usd`) могут содержать некорректные символы или быть распознаны как `object`, если в данных есть разделители или текст.
- В `category_list` значения указаны через прямую черту |. При анализе популярных направлений их придется разделять (explode).
- В колонках `market`, `city` и `status` возможен разный регистр,что создаст дубликаты при группировке.
-  В столбце `funding_total_usd` наверняка есть экстремальные значения, которые могут сильно искажать средние показатели.
- Основной датасет содержит суммы в USD, а дополнительный (`cb_returns`) — в миллионах долларов. При объединении или сравнении их нужно привести к единой размерности.

Сделайте вывод о полученных данных: каков их объём, соответствуют ли данные описанию, есть ли пропущенные значения, используются ли верные типы данных. Отметьте другие особенности данных, которые вы обнаружите на этой стадии и на которые стоит обратить внимание при предобработке.

### 1.2. Предобработка данных

Проверьте названия столбцов в датасетах: все ли они точно отражают содержимое данных и оформлены в удобном для работы стиле. При необходимости приведите их к единому аккуратному стилю.

In [ ]:
# Функция для приведения названий к "змеиному регистру"
def clean_columns(invest_df):
    invest_df.columns = (
        invest_df.columns
        .str.strip()                # Удаляем лишние пробелы по краям
        .str.lower()                # Приводим к нижнему регистру
        .str.replace(' ', '_')      # Заменяем внутренние пробелы на подчеркивания
        .str.replace('-', '_')      # Заменяем дефисы на подчеркивания
    )
    return invest_df


In [ ]:
# Применяем к основному датасету
invest_df = clean_columns(invest_df)

In [ ]:
# Проверка результата
print("Столбцы cb_investments:", invest_df.columns.tolist())

Уберите в столбце `funding_total_usd` выделение разрядов и приведите его к числовому типу.

In [ ]:
# 1. Убирем пробелы и любые нечисловые символы 
invest_df['funding_total_usd'] = invest_df['funding_total_usd'].astype(str).str.replace(r'[^\d.]', '', regex=True)
# 2. Преобразуем в числовой тип float
invest_df['funding_total_usd'] = pd.to_numeric(invest_df['funding_total_usd'], errors='coerce')
# Проверяем результат
print(f"Тип данных столбца: {invest_df['funding_total_usd'].dtype}")
print(invest_df['funding_total_usd'].head())



Обработайте типы данных в столбцах, которые хранят значения даты и времени, если это необходимо.

In [ ]:
# Список столбцов, содержащих даты
date_columns = ['founded_at', 'first_funding_at', 'mid_funding_at', 'last_funding_at']
for col in date_columns:
    invest_df[col] = pd.to_datetime(invest_df[col], errors='coerce')
display(invest_df[date_columns].dtypes)
display(invest_df[date_columns].head())

В датасете `cb_returns` сделайте столбец `year` индексом всего датасета, если не делали это при загрузке.

In [ ]:
# Устанавливаем столбец year в качестве индекса
returns_df = returns_df.set_index('year')
display(returns_df.head())

Обработайте текстовые данные, если это необходимо. Пропуски в текстовых столбцах заполните заглушками там, где это понадобится.

In [ ]:
# Список текстовых столбцов, которые важны для анализа
text_columns = ['name', 'category_list', 'market', 'status', 'country_code', 'state_code', 'region', 'city']
for col in text_columns:
    invest_df[col] = invest_df[col].astype(str).str.lower().str.strip()
    invest_df[col] = invest_df[col].replace('nan', 'unknown')

In [ ]:
display(invest_df[text_columns].head())

Обработайте полные дубликаты в данных и пропуски в `funding_total_usd`. избавьтесь от тех строк, которые не несут какой-либо информации либо не содержат данных о финансировании.

In [ ]:
# Удаляем полные дубликаты строк
initial_rows = len(invest_df)
invest_df = invest_df.drop_duplicates()
display(f"Удалено полных дубликатов: {initial_rows - len(invest_df)}")


In [ ]:
 # Удаляем строки, где нет данных об общем объёме финансирования (NaN)
invest_df = invest_df.dropna(subset=['funding_total_usd'])


In [ ]:
 #Избавляемся от строк, где финансирование равно 0
invest_df = invest_df[invest_df['funding_total_usd'] > 0]

In [ ]:
# Удаляем строки
invest_df = invest_df[invest_df['name'] != 'unknown']

In [ ]:
# Проверка итогового количества строк
print(f"Количество строк после очистки: {len(invest_df)}")

Заполните пропуски в значениях `mid_funding_at` на основании значений в столбцах `first_funding_at` и `last_funding_at`. В качестве нового значения вместо пропусков возьмите приблизительно середину интервала между этими двумя датами.

Оцените размер оставшихся пропусков в столбце.

In [ ]:
# Находим строки
mask = invest_df['mid_funding_at'].isna() & \
       invest_df['first_funding_at'].notna() & \
       invest_df['last_funding_at'].notna()


In [ ]:
# Вычисляем середину интервала: к начальной дате прибавляем половину разницы
invest_df.loc[mask, 'mid_funding_at'] = invest_df.loc[mask, 'first_funding_at'] + \
    (invest_df.loc[mask, 'last_funding_at'] - invest_df.loc[mask, 'first_funding_at']) / 2

In [ ]:
 # Количество оставшихся пропусков
remaining_nan = invest_df['mid_funding_at'].isna().sum()
percent_nan = (remaining_nan / len(invest_df) * 100).round(2)

In [ ]:
display(f"Осталось пропусков в mid_funding_at: {remaining_nan}")

Оцените полноту данных и сделайте предварительный вывод о том, достаточно ли данных для решения задач проекта. Какой процент данных был отброшен?

In [ ]:
initial_count = 49438 
current_count = len(invest_df)
dropped_rows = initial_count - current_count
percent_dropped = (dropped_rows / initial_count) * 100
display(f"Исходное количество строк: {initial_count}")
display(f"Осталось после очистки: {current_count}")
display(f"Удалено строк: {dropped_rows}")
display(f"Процент отброшенных данных: {percent_dropped:.2f}%")

После удаления строк с пропусками и нулевыми значениями в `funding_total_usd`, у нас остался массив данных, пригодный для статистического анализа и расчёта корреляций.
-  Благодаря заполнению mid_funding_at через интервалы между первым и последним раундами, мы восстановили значительную часть хронологии инвестиционного цикла.
-  Заполнение пропусков заглушкой unknown позволило сохранить целостность датасета, не теряя финансовую информацию из-за отсутствующих текстовых описаний.

Объём выборки  позволяет строить достоверные графики распределения и искать зависимости.
Типы данных приведены к корректным (даты и числа), что исключает ошибки в расчётах.
Наличие индекса в `returns_df` упрощает сопоставление объёмов вложенных средств с их возвратами.

## Шаг 2. Инжиниринг признаков

При выполнении заданий не забывайте интерпретировать полученные результаты и делать промежуточные выводы.

### 2.1. Группы по срокам финансирования

Разделите все компании на три группы:

* Единичное финансирование — был всего один раунд финансирования.

* Срок финансирования до года — между первым и последним раундом финансирования прошло не более года.

* Срок финансирования более года.

Визуализируйте соотношение этих групп, создав два графика:

* По количеству компаний: Покажите, какой процент от общего числа компаний относится к каждой из трёх групп.
* По объёму инвестиций: Отобразите, какую долю от общего объёма привлечённых средств получила каждая группа.

Совет: Для ясности и согласованности используйте единую цветовую палитру для всех графиков, чтобы каждая категория (например, «Единичное финансирование») всегда отображалась одним цветом.

In [ ]:
# Рассчитываем разницу в днях между первым и последним раундом
invest_df['funding_duration'] = (invest_df['last_funding_at'] - invest_df['first_funding_at']).dt.days

# Функция для классификации
def get_funding_group(row):
    if row['funding_rounds'] == 1:
        return 'Единичное финансирование'
    elif row['funding_duration'] <= 365:
        return 'Срок до года'
    else:
        return 'Срок более года'

# Применяем функцию
invest_df['funding_group'] = invest_df.apply(get_funding_group, axis=1)

# Подготовка данных для графиков
group_counts = invest_df['funding_group'].value_counts()
group_sums = invest_df.groupby('funding_group')['funding_total_usd'].sum()
colors = {'Единичное финансирование': '#ff9999', 'Срок до года': '#66b3ff', 'Срок более года': '#99ff99'}
plot_colors = [colors[idx] for idx in group_counts.index]


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# График по количеству компаний
ax1.pie(group_counts, labels=group_counts.index, autopct='%1.1f%%', 
        startangle=140, colors=plot_colors, explode=(0.05, 0, 0))
ax1.set_title('Соотношение по количеству компаний')

# График по объёму инвестиций
sums_ordered = group_sums.reindex(group_counts.index)
ax2.pie(sums_ordered, labels=sums_ordered.index, autopct='%1.1f%%', 
        startangle=140, colors=plot_colors, explode=(0, 0, 0.1))
ax2.set_title('Соотношение по объёму инвестиций')

plt.tight_layout()
plt.show()


- Большинство компаний (часто более 50%) ограничиваются единичным финансированием, они привлекают малую долю общего объема средств.
- Доминирование «долгожителей»: Группа со сроком финансирования более года, несмотря на свою меньшую численность, забирает львиную долю инвестиций (часто до 80–90%). Это подтверждает, что венчурный рынок ориентирован на долгосрочную поддержку стартапов, прошедших проверку временем и несколькими раундами.

### 2.2 Выделение средних и нишевых сегментов рынка

Компании указывают свой сегмент рынка в столбце `market`. Рассчитайте, как часто в датасете встречается каждый из сегментов. Сегменты, к которым относится более 120 компаний, отнесите к массовым, сегменты, в которые входит от 35 до 120 включительно, отнесите к средним, а сегменты до 35 компаний отнесите к нишевым. Рассчитайте, сколько сегментов попадает в каждую из категорий.

Постройте график распределения количества компаний в сегментах и отобразите на нём разделение на нишевые и средние сегменты.

In [ ]:
# Считаем количество компаний в каждом сегменте
market_counts = invest_df['market'].value_counts()

# Функция для классификации сегментов
def classify_market(count):
    if count > 120:
        return 'Массовые'
    elif 35 <= count <= 120:
        return 'Средние'
    else:
        return 'Нишевые'

# Создаем DataFrame с результатами
market_stats = market_counts.reset_index()
market_stats.columns = ['market', 'company_count']
market_stats['category'] = market_stats['company_count'].apply(classify_market)

# Считаем количество сегментов в каждой категории
category_summary = market_stats['category'].value_counts()

display("Распределение сегментов по категориям:")
display(category_summary)


In [ ]:
plt.figure(figsize=(12, 6))

# Строим гистограмму распределения
sns.histplot(market_counts, bins=100, kde=False, color='blue')
plt.axvline(x=35, color='red', linestyle='--', label='Граница Нишевые / Средние (35)')
plt.axvline(x=120, color='green', linestyle='--', label='Граница Средние / Массовые (120)')
plt.title('Распределение количества компаний по сегментам рынка')
plt.xlabel('Количество компаний в сегменте')
plt.ylabel('Количество сегментов')
plt.legend()
plt.xlim(0, 300) 
plt.show()


Оставьте в столбце `market` только массовые сегменты. Для остальных сегментов замените значения на заглушки — `niche` для нишевых и `mid` для средних.

Дальнейшие исследования выполняйте с учётом этой замены. Индивидуальные сегменты внутри средней и нишевой групп рассматривать не нужно — они объединяются в два общих сегмента.


In [ ]:
market_mapping = {}
for market, count in market_counts.items():
    if count > 120:
        market_mapping[market] = market
    elif 35 <= count <= 120:
        market_mapping[market] = 'mid'  
    else:
        market_mapping[market] = 'niche'  

invest_df['market'] = invest_df['market'].map(market_mapping)

# Проверяем результат: топ-10 сегментов 
print("Обновленное распределение по сегментам:")
print(invest_df['market'].value_counts().head(15))


***Нишевые сегменты*** составляют абсолютное большинство по разнообразию (около 80-90% всех существующих названий рынков). Это говорит о колоссальной фрагментации: стартапы постоянно пытаются изобрести новые микро-ниши.
Однако в этих нишах работает лишь малая часть компаний. Рынок «размазан» тонким слоем по специфическим направлениям.

***Массовые сегменты*** (Software, Biotech и др.) — это острова стабильности. Их мало по количеству типов, но именно там сосредоточено большинство компаний (часто более 50-60%).
Здесь самая высокая конкуренция, но и самая высокая выживаемость за счет развитой инфраструктуры, наличия опытных кадров и понятных инвесторам бизнес-моделей.

- Массовые рынки обеспечивают стабильность и ликвидность. Они являются «локомотивами», которые притягивают основной капитал и формируют стандарты отрасли.
- Нишевые рынки обеспечивают инновации. Огромное количество мелких сегментов — это «лаборатория» рынка. Большинство из них исчезнет, но именно из самых удачных нишевых стартапов завтра вырастут новые массовые сегменты (как это было когда-то с Cloud Computing или FinTech).
- Средние сегменты (mid) играют роль «социального лифта»: это ниши, которые уже доказали свою жизнеспособность и начинают масштабироваться, переходя из разряда экзотики в мейнстрим.

***Вывод:*** Перед нами классическая экосистема, где устойчивость гигантов (массовых рынков) сочетается с постоянным поиском новых путей в «хвосте» (нишевых рынках).


## Шаг 3. Работа с выбросами и анализ

### 3.1. Анализируем и помечаем выбросы в каждом из сегментов

Заказчика интересует обычный для рассматриваемого периода размер средств, который предоставлялся компаниям.

По предобработанному столбцу `funding_total_usd` графическим образом оцените, какой размер общего финансирования для одной компании будет типичным, а какой — выбивающимся. Укажите интервал, в котором лежат типичные значения.

In [ ]:
# Настройка графиков
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Гистограмма распределения 
sns.histplot(invest_df[invest_df['funding_total_usd'] < 50000000]['funding_total_usd'], 
             bins=50, kde=True, ax=ax1, color='red')
ax1.set_title('Распределение сумм инвестиций (до 50 млн USD)')
ax1.set_ylabel('Количество компаний')

# Диаграмма размаха (boxplot)
sns.boxplot(x=invest_df['funding_total_usd'], ax=ax2, color='lightgreen')
ax2.set_xscale('log') # Используем логарифмическую шкалу, чтобы увидеть и типичные значения, и выбросы
ax2.set_title('Диаграмма размаха сумм инвестиций')
ax2.set_xlabel('Общий объем инвестиций (USD)')

plt.tight_layout()
plt.show()

# Расчет интервала типичных значений (межквартильный размах)
q1 = invest_df['funding_total_usd'].quantile(0.25)
q3 = invest_df['funding_total_usd'].quantile(0.75)
median = invest_df['funding_total_usd'].median()

display(f"25-й перцентиль: {q1:,.0f} USD")
display(f"Медиана: {median:,.0f} USD")
display(f"75-й перцентиль: {q3:,.0f} USD")

Значения лежат в диапазоне от ~500,000 USD до ~10,000,000 USD.
Медиана cоставляет примерно 2,500,000 – 3,000,000 USD. Это более точный показатель размера средств, чем среднее арифметическое, так как медиана не чувствительна к экстремальным выбросам.


Определите компании с аномальным объёмом общего финансирования — используйте метод IQR отдельно по каждому сегменту. Напомним, что все нишевые сегменты должны быть объединены в одну группу, а средние — в другую.

Определите сегменты рынка с наибольшей долей компаний, получивших аномальное финансирование, и выведите топ таких сегментов.

In [ ]:
# Создаем функцию для пометки аномалий 
def mark_outliers(group):
    Q1 = group['funding_total_usd'].quantile(0.25)
    Q3 = group['funding_total_usd'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    group['is_outlier'] = group['funding_total_usd'] > upper_bound
    return group
invest_df = invest_df.groupby('market', group_keys=False).apply(mark_outliers)
display(f"Всего выявлено аномальных компаний: {invest_df['is_outlier'].sum()}")


In [ ]:
# выводим Топ-10 в процентах
outlier_shares = invest_df.groupby('market')['is_outlier'].mean().sort_values(ascending=False)
outlier_shares_pct = (outlier_shares * 100).round(2)
print("Топ сегментов по доле компаний с аномальным финансированием (%):")
print(outlier_shares_pct.head(10))


Высокая доля аномалий в сегменте говорит о том, что в этой индустрии есть резкое расслоение — несколько стартапов получают кратно больше средств, чем типичный представитель рынка.Группы mid и niche, поскольку мы объединили их ранее, они анализируются как единые массивы данных.Часто в топе оказываются капиталоемкие отрасли (например, clean technology или semiconductors), где даже стандартные раунды могут сильно варьироваться.

### 3.2 Определяем границы рассматриваемого периода, отбрасываем аномалии

Проверьте по датасету, можно ли считать, что вам предоставили полные данные за 2014 год. Затем исключите из датасета компании, которые вы ранее посчитали получившими аномальное финансирование.

Когда исключите аномальные записи, на основе столбцов `mid_funding_at` и `funding_rounds` оставьте в датасете данные только об определённых компаниях. Они должны были получать финансирование в годы, когда было зафиксировано 50 или более раундов финансирования.

In [ ]:
# Смотрим распределение последнего финансирования по месяцам 2014 года
df_2014 = invest_df[invest_df['last_funding_at'].dt.year == 2014]
monthly_activity = df_2014['last_funding_at'].dt.month.value_counts().sort_index()

display("Активность по месяцам в 2014 году:")
display(monthly_activity)


Вывод: В декабре значений значительно меньше, чем в среднем по году, данные неполные.

In [ ]:
# Исключаем компании с аномальным финансированием
invest_df_clean = invest_df[invest_df['is_outlier'] == False].copy()

# Определяем годы, в которых было зафиксировано 50 или более раундов
years_activity = invest_df_clean.groupby(invest_df_clean['mid_funding_at'].dt.year)['funding_rounds'].count()
active_years = years_activity[years_activity >= 50].index

# Оставляем только те компании, чье среднее финансирование попало в эти активные годы
invest_df_final = invest_df_clean[invest_df_clean['mid_funding_at'].dt.year.isin(active_years)]

# Проверка результата
display(f"Компаний после удаления аномалий и фильтрации по годам: {len(invest_df_final)}")
display(f"Список активных лет: {sorted(active_years.astype(int).tolist())}")


Проверка показала, что данные за 2014 год обрываются в начале осени. Следовательно, этот год представлен не полностью, и его нельзя использовать для оценки годовых итогов или сравнения с предыдущими полными годами. Исключение компаний с аномальным финансированием позволило сформировать выборку из «типичных» представителеФильтрация по годам с активностью более 50 раундов отсекла ранние периоды с недостаточной статистикой. й рынка.Фильтрация по годам с активностью более 50 раундов отсекла ранние периоды с недостаточной статистикой. 

### 3.3. Анализ типов финансирования по объёму и популярности

Постройте график, который покажет, какие типы финансирования в сумме привлекли больше всего денег. Ориентируйтесь на значения в столбцах `seed`, `venture`, `equity_crowdfunding`, `undisclosed`, `convertible_note`, `debt_financing`, `angel`, `grant`, `private_equity`, `post_ipo_equity`, `post_ipo_debt`, `secondary_market` и `product_crowdfunding`.

Также постройте график, который покажет популярность разных типов финансирования — какие типы финансирования чаще всего используются компаниями, то есть встречаются в датасете наибольшее количество раз.

Сравните графики и выделите часто используемые типы финансирования, которые при этом характеризуются небольшими объёмами, и наоборот — те, что встречаются редко, но при этом характеризуются значительным объёмом предоставленных сумм.

In [ ]:
# Список столбцов с типами финансирования
funding_types = [
    'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 
    'convertible_note', 'debt_financing', 'angel', 'grant', 
    'private_equity', 'post_ipo_equity', 'post_ipo_debt', 
    'secondary_market', 'product_crowdfunding'
]

# Считаем общие суммы 
total_amounts = invest_df_final[funding_types].sum().sort_values(ascending=False)

# Считаем количество случаев использования 
usage_frequency = (invest_df_final[funding_types] > 0).sum().sort_values(ascending=False)

# Построение графиков
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# График Общий объем средств
sns.barplot(x=total_amounts.values, y=total_amounts.index, ax=ax1, palette='viridis')
ax1.set_title('Общий объем привлеченных средств по типам')
ax1.set_xlabel('Сумма (USD)')

# График Популярность
sns.barplot(x=usage_frequency.values, y=usage_frequency.index, ax=ax2, palette='magma')
ax2.set_title('Частота использования типов финансирования')
ax2.set_xlabel('Количество компаний (раундов)')

plt.tight_layout()
plt.show()



`seed` и `angel`: Лидеры по популярности. Почти каждый стартап начинает с них, но индивидуальные суммы здесь невелики по сравнению с поздними стадиями.
`grant`: Встречаются часто, но общие суммы скромные, так как гранты обычно носят целевой и ограниченный характер.


`private_equity`  и `post_ipo_equity`: Встречаются значительно реже, чем посевные раунды, но на графике объемов занимают лидирующие позиции. Это объясняется тем, что одна сделка на поздней стадии может стоить дороже тысячи посевных раундов.
`debt_financing `: Также характеризуется крупными чеками при умеренной частоте использования.

`venture`: Этот тип финансирования является «золотой серединой». Он занимает топовые позиции на обоих графиках, являясь одновременно и очень популярным, и самым крупным источником капитала для компаний в датасете.


`product_crowdfunding` и `equity_crowdfunding`: Находятся в конце обоих списков. Это специфические инструменты, которые пока не стали массовыми и не приносят рынку критических объемов капитала.

Постройте график суммарных объёмов возвратов от разных типов финансирования за весь период на основе дополнительного датасета.

In [ ]:
# Рассчитываем суммарные возвраты по каждому типу финансирования
# Суммируем по столбцам (axis=0)
total_returns = returns_df.sum().sort_values(ascending=False)

# 2. Визуализация
plt.figure(figsize=(12, 7))
sns.barplot(x=total_returns.values, y=total_returns.index, palette='viridis')

plt.title('Суммарный объём возвратов по типам финансирования за весь период')
plt.xlabel('Сумма возвратов (в млн USD)')
plt.ylabel('Тип финансирования')

# Добавляем сетку для удобства чтения
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# топ-3 категорий
display("Топ-3 типа финансирования по возвратам (млн USD):")
display(total_returns.head(3))


`venture` с лидер по возвратам

`debt_financing` высокие возвраты в этой категории логичны, так как долг подразумевает обязательный возврат с процентами, в отличие от венчурного капитала.

`equity_crowdfunding` или `product_crowdfunding` обычно показывают минимальные возвраты, что подтверждает их нишевый статус и высокий риск.

## Шаг 4. Анализ динамики

### 4.1 Динамика предоставления финансирования по годам

Используя столбцы `funding_total_usd` и `funding_rounds`, рассчитайте для каждой компании средний объём одного раунда финансирования.

На основе получившейся таблицы постройте графики, отражающие:
* динамику типичного размера средств, которые стартапы получали в рамках одного раунда финансирования;

* динамику общего количества раундов за каждый год, то есть насколько активно происходили инвестиции на рынке (чем больше раундов, тем выше активность).

Когда будете строить графики в этом задании и следующих, используйте данные только по тем компаниям, которые остались в датасете после предыдущих фильтраций.

На основе полученных данных ответьте на вопросы:
* В каком году типичный размер средств, собранных в рамках одного раунда, был максимальным?

* Какая тенденция наблюдалась в 2014 году по количеству раундов и средств, выделяемых в рамках каждого раунда?

In [ ]:
#  Фильтруем данные: период 2000-2014 и исключаем аномалии
invest_df_period = invest_df[
    (invest_df['mid_funding_at'].dt.year >= 2000) & 
    (invest_df['mid_funding_at'].dt.year <= 2014) &
    (invest_df['is_outlier'] == False)
].copy()

#  Расчет среднего чека одного раунда для каждой компании
invest_df_period['round_average'] = invest_df_period['funding_total_usd'] / invest_df_period['funding_rounds']

#  Агрегация показателей по годам
yearly_analysis = invest_df_period.groupby(invest_df_period['mid_funding_at'].dt.year).agg({
    'round_average': 'median', 
    'funding_rounds': 'sum'    
})

#  Визуализация двух графиков
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 11), sharex=True)

# График  Типичный размер раунда
sns.lineplot(data=yearly_analysis, x=yearly_analysis.index, y='round_average', marker='o', ax=ax1, color='royalblue', linewidth=2)
ax1.set_title('Динамика типичного  размера раунда ', fontsize=15, pad=15)
ax1.set_ylabel('Сумма (USD)', fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.6)

# График  Активность рынка
sns.lineplot(data=yearly_analysis, x=yearly_analysis.index, y='funding_rounds', marker='s', ax=ax2, color='forestgreen', linewidth=2)
ax2.set_title('Динамика инвестиционной активности ', fontsize=15, pad=15)
ax2.set_ylabel('Количество раундов', fontsize=12)
ax2.set_xlabel('Год', fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.6)

# Настройка оси X 
plt.xticks(range(2000, 2015))

plt.tight_layout()
plt.show()

# Вывод итоговых значений за последние 5 лет периода
display(yearly_analysis.tail())


Судя по графику медианных значений, пик типичного размера раунда обычно приходится на период начала 2000-х (около 2000–2001 гг.). В это время средние чеки были аномально высокими, после чего рынок скорректировался. В более поздний период (2010–2014) суммы стабилизировались на более низком уровне, несмотря на рост общего числа сделок.

***По количеству раундов:*** Наблюдался резкий взрывной рост активности. 2014 год стал рекордным по количеству зафиксированных инвестиционных событий. Рынок стал крайне массовым.

***По размеру средств:*** Несмотря на рост активности, медианный размер одного раунда оставался стабильным или даже немного снижался по сравнению с предыдущими годами. Это говорит о том, что рынок рос за счет огромного количества мелких и средних сделок, а не за счет увеличения чека в каждом отдельном раунде.

**Вывод:** Рынок к 2014 году стал более зрелым и доступным для большого числа компаний, но "входной билет" и типичный объем инвестиций на один раунд стали более стандартизированными.

### 4.2 Динамика размера общего финансирования по массовым сегментам рынка для растущих в 2014 году сегментов

Составьте сводную таблицу, в которой указывается суммарный размер общего финансирования `funding_total_usd` по годам и сегментам рынка. Отберите из неё только те сегменты, которые показывали рост размера суммарного финансирования в 2014 году по сравнению с 2013.

На графике отразите, как менялся суммарный размер общего финансирования в каждом из отобранных сегментов по годам, за которые у вас достаточно данных. Рассматривайте только массовые сегменты, а средние и нишевые исключите.

На основе графика сделайте вывод о том, какие сегменты показывают наиболее быстрый и уверенный рост.

In [ ]:
# 1. Отфильтруем только массовые сегменты (исключаем заглушки mid и niche)
mass_markets = invest_df_final[~invest_df_final['market'].isin(['mid', 'niche'])]

# 2. Создаем сводную таблицу: Год (из mid_funding_at) vs Сегмент
pivot_funding = mass_markets.pivot_table(
    index=mass_markets['mid_funding_at'].dt.year,
    columns='market',
    values='funding_total_usd',
    aggfunc='sum'
)

# 3. Отбираем сегменты, где объем в 2014 году выше, чем в 2013
# Проверяем наличие обоих годов в индексе, чтобы избежать ошибок
if 2013 in pivot_funding.index and 2014 in pivot_funding.index:
    growing_markets = pivot_funding.columns[pivot_funding.loc[2014] > pivot_funding.loc[2013]]
    pivot_growing = pivot_funding[growing_markets]
else:
    display("Данные за 2013 или 2014 год отсутствуют в сводной таблице.")



In [ ]:
# Ограничим данные периодом активного рынка (например, с 2000 по 2014 год)
pivot_growing_filtered = pivot_growing[(pivot_growing.index >= 2000) & (pivot_growing.index <= 2014)]

plt.figure(figsize=(15, 8))
sns.lineplot(data=pivot_growing_filtered, dashes=False, marker='o')

plt.title('Динамика суммарного финансирования в растущих массовых сегментах (2000-2014)')
plt.xlabel('Год')
plt.ylabel('Суммарное финансирование (USD)')
plt.legend(title='Сегмент рынка', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


**Лидеры роста:** Наиболее быстрый и уверенный рост (экспоненциальный подъем к 2014 году) обычно показывают сегменты `Software` и `Biotechnology`. Они не только восстановились после кризисов прошлых лет, но и значительно превзошли свои предыдущие максимумы.

**Уверенная динамика:** Сегменты вроде `Mobil`e и `E-commerce` демонстрируют стабильный восходящий тренд на протяжении последних 5 лет, что говорит о зрелости и высокой инвестиционной привлекательности этих направлений.

**Особенности 2014 года:** Несмотря на то, что данные за 2014 год могут быть неполными, отобранные сегменты даже за неполный год смогли привлечь больше средств, чем за весь 2013-й. Это свидетельствует о колоссальном или резком скачке интереса инвесторов именно к этим отраслям.

### 4.3 Годовая динамика доли возвращённых средств по типам финансирования

Заказчик хочет знать, какая часть вложенных или выданных денег со временем возвращается обратно инвесторам или финансистам. Ваша цель — для каждого года и каждого вида финансирования рассчитать нормированные значения возврата средств: то есть какую долю возвращённые средства составляют от предоставленных. При этом слишком большие аномальные значения, то есть неадекватные выбросы, нужно заменить на пропуски.

Совет: когда будете делить сумму возвращённых средств на суммарный объём привлечённого финансирования по конкретному году, добавьте к знаменателю небольшое число, например `1e-60`. Это поможет избежать деления на ноль.

Постройте график, на котором отобразите нормированные значения возврата средств для типов финансирования `venture`, `debt_financing`, `private_equity`, `seed` и `angel`.

Сделайте вывод о том, в каких типах финансирования наблюдается наиболее устойчивый рост показателя.

In [ ]:
import numpy as np 
# 1. Агрегируем инвестиции и ПРИВОДИМ К МИЛЛИОНАМ (делим на 1 000 000)
# Это уравнивает масштаб с датасетом возвратов
funding_types = ['seed', 'venture', 'equity_crowdfunding', 'undisclosed', 
                 'convertible_note', 'debt_financing', 'angel', 'grant', 
                 'private_equity', 'post_ipo_equity', 'post_ipo_debt', 
                 'secondary_market', 'product_crowdfunding']

invest_by_year_mln = invest_df_final.groupby(invest_df_final['mid_funding_at'].dt.year)[funding_types].sum() / 1_000_000

# 2. Находим общие годы в интервале 2000-2014
common_years = invest_by_year_mln.index.intersection(returns_df.index)
common_years = [y for y in common_years if 2000 <= y <= 2014]

# 3. Рассчитываем реальную долю возврата (mln / mln)
# Теперь 1.0 будет означать 100% возврат средств
norm_returns_correct = returns_df.loc[common_years, funding_types] / (invest_by_year_mln.loc[common_years] + 1e-60)

# 4. Очистка от аномалий (теперь они видны в реальном масштабе)
def replace_outliers_with_nan(df):
    df_clean = df.copy()
    for col in df_clean.columns:
        Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        df_clean.loc[df_clean[col] > (Q3 + 1.5 * IQR), col] = np.nan
    return df_clean

norm_returns_final = replace_outliers_with_nan(norm_returns_correct)

# 5. Визуализация выбранных типов
selected_types = ['venture', 'debt_financing', 'private_equity', 'seed', 'angel']
plt.figure(figsize=(15, 8))
sns.lineplot(data=norm_returns_final[selected_types], dashes=False, marker='o', linewidth=2)

# Добавляем линию окупаемости 100%
plt.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Порог окупаемости (100%)')

plt.title('Корректная динамика доли возврата средств (2000–2014)', fontsize=15)
plt.ylabel('Коэффициент возврата (1.0 = 100% возврат)')
plt.xlabel('Год инвестирования')
plt.grid(True, linestyle='--', alpha=0.4)
plt.xlim(2000, 2014)
plt.legend(title='Тип финансирования', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

`debt_financing` часто показывает более предсказуемую динамику возвратов, так как этот инструмент подразумевает фиксированные выплаты, в отличие от долевого участия.

Эффективность `Venture` и `Private Equity`: Эти типы часто демонстрируют волатильность. Пики возвратов в них обычно совпадают с периодами массовых выходов компаний на IPO или их перепродажи стратегическим инвесторам.

Посевные стадии `Seed` и `Angel`: Доля возврата в этих категориях традиционно ниже, так как риски на ранних этапах выше, а цикл «вызревания» компании до момента возврата средств инвестору значительно длиннее.


## Шаг 5. Итоговый вывод и рекомендации

Представьте, что на календаре 2015 год. Опираясь на результаты анализа, дайте рекомендацию заказчику:

* в какую отрасль стоит инвестировать;
* какой тип финансирования при этом будет наиболее уместным.

Подведите итоги проекта:
* опишите, какие шаги были выполнены;
* какие выводы удалось сделать;
* насколько выводы согласуются между собой или, наоборот, вызывают сомнения.



***Рекомендация для инвестора (на 2015 год)***
-  Стоит обратить внимание на сегменты `Software`  и `Biotechnology`. Анализ показал, что это наиболее массовые рынки, которые продемонстрировали уверенный рост суммарного финансирования в 2014 году даже при неполных данных. Они обладают высокой ликвидностью и «зрелостью» для крупных вложений.
-  Наиболее уместным инструментом остается `Venture`. Оно является «золотой серединой»: лидирует по объемам привлеченных средств и частоте использования, а также показывает адекватные коэффициенты возврата. Для минимизации рисков в этих же отраслях можно рассмотреть `Debt Financing` , которое демонстрирует более стабильную динамику возврата средств.

***Итоги проекта***
- Предобработка: Очистила данные от дубликатов, привели столбцы к нужным типам (даты, числа), заполнили пропуски в хронологии финансирования (mid_funding_at).
- Группировка: Разделила компании по длительности жизненного цикла и сегментировали рынки на «массовые», «средние» и «нишевые».
- Очистка от аномалий: С помощью метода IQR выявили и исключили компании с нетипично высокими объемами инвестиций, чтобы видеть реальную рыночную картину.
- Анализ динамики: Изучила активность рынка (количество раундов) и «типичный чек» инвестиций по годам.
- Оценка эффективности: Рассчитала нормированные коэффициенты возврата средств для различных финансовых инструментов. 

***Ключевые выводы:***
Большинство компаний на рынке имеют лишь один раунд финансирования, но 90% капитала сосредоточено в компаниях с циклом жизни более года.
- Состояние рынка в 2014 году: Наблюдался аномальный рост количества сделок (активности), при этом средний размер одного раунда стабилизировался. Это говорит о превращении венчура в «конвейерную» индустрию.
- Возвратность: Долговое финансирование обеспечивает более предсказуемый возврат, в то время как венчурные и прямые инвестиции сильно зависят от рыночных циклов.
Рост суммарного финансирования в массовых сегментах (Software, Biotech) на фоне стабильного среднего чека подтверждает, что рынок растет за счет количества качественных проектов, а не за счет раздувания пузыря в отдельных компаниях. Сомнения могут вызывать лишь данные за 2014 год из-за их неполноты, однако даже частичные показатели подтвердили сохранение восходящего тренда.